<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# Statistics for machine learning — Code-Along
---
<small>© COPYRIGHT 2026 DECODED LIMITED. ALL RIGHTS RESERVED. Web: https://decoded.com/ </small>


### Programmes

* Data Science Accelerator Workshop - Introduction to Statistics

### Learning Objectives

* Explain the purpose and structure of hypothesis testing including p-values and interpret the results
* Understand the danger of averages and why reporting only the mean can hide critical insights about inequality and risk
* Describe Simpson's paradox to show how data reasoning can go wrong
* Assess normality using histograms, Q-Q plots, and P-P plots to spot non-normal distributions
* Conduct formal normality test by applying Shapiro-Wilk and other statistical tests for normality
* Fix skewed and heavy-tailed distributions for better model performance
* Understanding why normality assumptions matter for regression, classification, and hypothesis testing






We use the same university datasets throughout:

* `university_performance.csv`
* `university_support_lookup.csv`

By the end, you will have practised:

* reading mathematical notation in plain English
* estimating probabilities from data
* calculating statistical measures in Python
* running and interpreting hypothesis tests
* checking correlation without claiming causation
* checking whether a grouped comparison may be affected by Simpson's paradox

---
### 🧑‍💻 How to use this notebook

Each section has:
- **📖 Concept** — a brief explanation of what you will do
- **🏋️ Your Turn** — an empty (or starter) code cell for you to complete
- **💡 Hint** — click the arrow below any hint block to reveal guidance




## 1. Code-along setup

We begin with the imports and a small helper function for finding files.

> **💬 Think about it:** Why do we use a helper function instead of hard-coding one file path?





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Standard imports for data science work
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import chi2, chi2_contingency

# matplotlib may not be installed in all environments
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print("matplotlib is not installed.")
```

**Tips:**
- `pathlib.Path` handles file paths across operating systems (Windows, Mac, Linux)
- Wrapping optional imports in `try/except` makes code more portable
- `scipy.stats` contains the hypothesis testing functions we will use

</details>

In [ ]:
# 🏋️ YOUR TURN
# Import the libraries you need for this session.
# You need: pathlib.Path, numpy (as np), pandas (as pd),
#           scipy.stats, chi2 and chi2_contingency from scipy.stats,
#           and matplotlib.pyplot (as plt) — wrap it in a try/except



## 2. Load the two datasets

Load the support lookup and the performance data.

> **💬 Pause and ask:** What does one row represent in each file?





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Use pd.read_csv() to load each file.
# If the files are in the same folder, you can write:
df_perf    = pd.read_csv("university_performance.csv")
df_support = pd.read_csv("university_support_lookup.csv")

# If the files are in a different folder, use Path to build the full path:
# data_path = Path("../data")
# df_perf   = pd.read_csv(data_path / "university_performance.csv")

# Quick check
df_perf.head()
```

**Tips:**
- `df.head()` shows the first 5 rows — use it after loading to verify the data loaded correctly
- `df.shape` shows (rows, columns) — use it to check the size of the dataset
- `df.dtypes` shows the data type of each column

</details>

In [ ]:
# 🏋️ YOUR TURN
# Load the two CSV files using pd.read_csv()
# Store them as df_perf and df_support



In [ ]:
# 🏋️ YOUR TURN
# Inspect the columns: what are the column names and data types?
# Try: .columns, .dtypes, .info(), .head()



In [ ]:
# 🏋️ YOUR TURN
# Check for missing values in both datasets
# Tip: use df.isnull().sum() to count missing values per column



## 3. Clean column names

Some column names have spaces. We clean them so the rest of the code is easier to read.

> **💬 Learner check:** Which original columns do you expect to change?





<details>
<summary>💡 Hint — click to expand</summary>

```python
# A helper function that strips whitespace, lowercases, and replaces spaces with underscores
def clean_columns(df):
    df = df.copy()                          # don't mutate the original
    df.columns = (
        df.columns
        .str.strip()                        # remove leading/trailing spaces
        .str.lower()                        # make lowercase
        .str.replace(" ", "_", regex=False) # replace spaces with underscores
    )
    return df

# Apply the function to both dataframes
df_perf    = clean_columns(df_perf)
df_support = clean_columns(df_support)
print(df_perf.columns.tolist())
```

**Tips:**
- `df.copy()` prevents modifying the original DataFrame in-place (avoids `SettingWithCopyWarning`)
- Chaining `.str` methods lets you transform all column names in one expression
- Always print the columns after cleaning to confirm the changes worked

</details>

In [ ]:
# 🏋️ YOUR TURN
# Define the clean_columns() function.
# It should: strip whitespace, lowercase, and replace spaces with underscores.

def clean_columns(df):
    # YOUR CODE HERE
    pass


In [ ]:
# 🏋️ YOUR TURN
# Apply clean_columns() to df_perf and df_support
# Print the column names after cleaning to verify



## 4. Join the lookup table

Join regional support information onto the main university performance table.

> **💬 Think about it:** What column do both tables share? That is the key for the join.






<details>
<summary>💡 Hint — click to expand</summary>

```python
# Use pd.merge() to join two DataFrames on a shared key column
# 'how' controls the join type: 'left' keeps all rows from df_perf
df = pd.merge(
    df_perf,        # left table
    df_support,     # right table (lookup)
    on="university_id",  # shared key column — check your column names!
    how="left"
)

print(df.shape)
df.head()
```

**Tips:**
- A **left join** keeps all rows from the left table even if there's no match on the right
- After joining, check `df.shape` — the row count should equal `df_perf.shape[0]`
- Check for NaN values after joining: `df.isnull().sum()`

</details>

In [ ]:
# 🏋️ YOUR TURN
# Merge df_perf with df_support on the shared key column
# Use a left join so all performance rows are kept
# Store the result as 'df'



In [ ]:
# 🏋️ YOUR TURN
# Perform data quality checks on the merged dataframe:
#   - How many rows and columns?
#   - Any missing values introduced by the join?
#   - Do the column names look correct?



## 5. Create useful rates

Raw counts are useful, but rates are easier to compare across universities of different sizes.

We create:

* **acceptance rate** = applications accepted / total applications
* **enrolment rate** = enrolled / accepted
* **high/low acceptance group** (above or below the median acceptance rate)
* **high/low spend group** (above or below median spend per student)





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Creating new columns in pandas is done by direct assignment
df["acceptance_rate"] = df["apps_accepted"] / df["apps_received"]
df["enrolment_rate"]  = df["new_students_enrolled"] / df["apps_accepted"]

# Create binary group columns using the median as the cut-off
# np.where(condition, value_if_true, value_if_false)
median_accept = df["acceptance_rate"].median()
df["high_acceptance"] = np.where(df["acceptance_rate"] > median_accept, 1, 0)

median_spend = df["expenditure_per_student"].median()
df["high_spend"] = np.where(df["expenditure_per_student"] > median_spend, 1, 0)

df[["acceptance_rate", "enrolment_rate", "high_acceptance", "high_spend"]].head()
```

**Tips:**
- Always check for division-by-zero: if a column can be 0, use `df["col"].replace(0, np.nan)` before dividing
- Use `pd.cut()` if you want to split into more than two groups (e.g., low/medium/high)
- `np.where()` is the pandas-friendly equivalent of an if-else statement applied to an entire column

</details>

In [ ]:
# 🏋️ YOUR TURN
# Create the following new columns on df:
#   1. acceptance_rate  = apps accepted / apps received
#   2. enrolment_rate   = new students enrolled / apps accepted
#   3. high_acceptance  = 1 if acceptance_rate > median, else 0
#   4. high_spend       = 1 if expenditure_per_student > median, else 0

# Check the column names in df first!
# print(df.columns.tolist())



## 6. Notation: translate symbols into Python

Notation is a compact way to describe what we already do in code.

* $X$ is a variable, here graduation rate.
* $E[X]$ is the expected value, estimated with the mean.
* $P(A)$ is the probability of event A.







## The Language of Statistical Inference

As we transition from Data Analysis to Data Science, we move from **descriptive** statistics (summarising the past) to **probabilistic** modelling (predicting the future). This requires a shift from talking about "the data" to talking about "the distribution".




### From mathematical notation to Python implementation

Throughout your data science career, you'll encounter mathematical notation in research papers, textbooks, and statistical libraries. This section helps you **translate notation into Python**.

| Mathematical Notation | English | Python Example |
|---|---|---|
| $E[X]$ | Expected value (theoretical mean) | `np.mean(X)` or `np.random.normal().mean()` |
| $X \sim N(\mu, \sigma^2)$ | X is normally distributed with mean μ and variance σ² | `np.random.normal(loc=μ, scale=σ, size=n)` |
| $P(A\|B)$ | Probability of A given B (conditional probability) | `len(A[B]) / len(B)` or `conditional_prob(A, B)` |
| $\sum_{i=1}^{n} x_i$ | Sum of all x values from index 1 to n | `np.sum(x)` or `sum(x)` |

### Why You Need This Notation

- **Reading research**: Papers on ML algorithms use $E[X]$ instead of "expected value"
- **Communicating findings**: Stakeholders trust clear mathematical formulation
- **Understanding libraries**: SciPy and scikit-learn documentation uses this notation extensively




<details>
<summary>💡 Hint — click to expand</summary>

```python
# E[X] — expected value (mean)
expected_value = df["graduation_rate"].mean()

# Median — robust centre measure
median_value   = df["graduation_rate"].median()

# σ (sigma) — standard deviation (spread)
std_value      = df["graduation_rate"].std()

print(f"E[X] (mean)   : {expected_value:.4f}")
print(f"Median        : {median_value:.4f}")
print(f"Std deviation : {std_value:.4f}")
```

**Tips:**
- `mean` is sensitive to outliers; `median` is robust — compare them to detect skew
- If `mean > median`, the distribution is right-skewed (a long tail on the right)
- If `mean < median`, the distribution is left-skewed (a long tail on the left)

</details>

In [ ]:
# 🏋️ YOUR TURN
# Calculate and print the following for the graduation_rate column:
#   - E[X]  : the expected value (mean)
#   - Median
#   - Standard deviation
#
# Bonus: compare mean vs median — what does the difference tell you?



## 7. Probability: estimate P(A)

Let event **A** mean: *a university has a strong outcome*.

Because `strong_outcome` is coded as 1 or 0, the mean gives the proportion of strong outcomes.

$$P(A) = \frac{\text{number of strong outcomes}}{\text{total number of universities}}$$





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Since strong_outcome is 1 (strong) or 0 (not strong),
# the mean gives us P(strong outcome)
p_strong = df["strong_outcome"].mean()

# Alternatively, count directly:
p_strong_manual = (df["strong_outcome"] == 1).sum() / len(df)

print(f"P(strong outcome) = {p_strong:.4f}")
print(f"That is {p_strong * 100:.1f}% of universities")
```

**Tips:**
- For a binary 0/1 column, `col.mean()` is equivalent to `col.sum() / len(col)` — the proportion of 1s
- Think of this as the *base rate* — the probability without any conditioning

</details>

In [ ]:
# 🏋️ YOUR TURN
# Calculate P(A): the probability that a university has a strong outcome.
#
# Recall: strong_outcome is a binary column (1 = strong, 0 = not strong)
# Two approaches:
#   1. Use .mean() on the binary column
#   2. Count rows where strong_outcome == 1 and divide by total rows
#
# Print the probability as a percentage.



## 8. Conditional probability: estimate P(A | B)

Now let **B** mean: *the university is private*.

$P(A \| B)$ means the probability of a strong outcome, given that we already know the university is private.

$$P(A | B) = \frac{P(A \cap B)}{P(B)}$$

In practice: **filter** the dataset to rows where B is true, then calculate P(A) on that subset.





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Step 1: Filter to only private universities (where B is true)
private     = df[df["private"] == 1]
non_private = df[df["private"] == 0]

# Step 2: Calculate P(A|B) = mean of strong_outcome within each group
p_strong_given_private     = private["strong_outcome"].mean()
p_strong_given_non_private = non_private["strong_outcome"].mean()

print(f"P(strong | private)     = {p_strong_given_private:.4f}")
print(f"P(strong | non-private) = {p_strong_given_non_private:.4f}")
```

**Tips:**
- Boolean indexing `df[df['col'] == value]` is the standard way to filter rows in pandas
- Compare P(A|B) with P(A) — if they differ significantly, B is informative about A
- This is the intuition behind feature importance in classification models!

</details>

In [ ]:
# 🏋️ YOUR TURN
# Calculate P(strong outcome | private) and P(strong outcome | non-private)
#
# Steps:
#   1. Filter df to rows where private == 1  → call it 'private_df'
#   2. Filter df to rows where private == 0  → call it 'non_private_df'
#   3. Compute the mean of strong_outcome in each group
#   4. Print both probabilities and compare them to P(A) from the previous cell



## 9. Statistical measures

Calculate centre and spread for the main numeric columns.

> **💬 Interpretation prompt:** Which measure would you explain first to a non-technical stakeholder?

The **four moments** of a distribution:
| Moment | What it measures | Python |
|--------|-----------------|--------|
| Mean | Centre | `df.mean()` |
| Variance / Std | Spread | `df.std()` |
| Skewness | Asymmetry | `df.skew()` |
| Kurtosis | Tail heaviness | `df.kurtosis()` |





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Select only numeric columns
num_cols = ["graduation_rate", "expenditure_per_student", "acceptance_rate"]

# Build a summary table using pandas aggregation
summary = df[num_cols].agg(["mean", "median", "std", "skew", "kurtosis"])

# Add kurtosis separately if needed
# summary.loc["kurtosis"] = df[num_cols].kurtosis()

summary.round(4)
```

**Interpreting skewness and kurtosis:**
- **Skewness ≈ 0**: symmetric (normal-like)
- **Skewness > 0**: right-skewed (long right tail) — mean > median
- **Skewness < 0**: left-skewed (long left tail) — mean < median
- **Kurtosis > 3** (excess > 0): heavier tails than normal (leptokurtic)
- **Kurtosis < 3** (excess < 0): lighter tails than normal (platykurtic)

</details>

In [ ]:
# 🏋️ YOUR TURN
# Create a statistical summary table for the key numeric columns.
# Include: mean, median, std, skewness, kurtosis
#
# Tip: df[cols].agg([...]) lets you compute multiple aggregations at once

num_cols = ["graduation_rate"]  # add more column names here



## 10. Visualise spread

Use histograms to check the shape of key variables.

> **💬 Pause and ask:** What can a histogram show that a mean cannot?





<details>
<summary>💡 Hint — click to expand</summary>

```python
if plt:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram of graduation_rate
    axes[0].hist(df["graduation_rate"].dropna(), bins=30, color="steelblue", edgecolor="white")
    axes[0].axvline(df["graduation_rate"].mean(),   color="red",    linestyle="--", label="mean")
    axes[0].axvline(df["graduation_rate"].median(), color="orange", linestyle="--", label="median")
    axes[0].set_title("Graduation Rate Distribution")
    axes[0].set_xlabel("Graduation Rate")
    axes[0].legend()

    # Histogram of expenditure_per_student
    axes[1].hist(df["expenditure_per_student"].dropna(), bins=30, color="coral", edgecolor="white")
    axes[1].set_title("Expenditure per Student Distribution")
    axes[1].set_xlabel("Expenditure per Student")

    plt.tight_layout()
    plt.show()
```

**Tips:**
- Adding vertical lines for mean and median helps visualise skewness
- `bins=30` is a good starting point — try 15 and 50 to see how bin width affects the picture
- `plt.tight_layout()` prevents overlapping labels between subplots

</details>

In [ ]:
# 🏋️ YOUR TURN
# Plot a histogram of graduation_rate.
# Add vertical dashed lines for the mean and median.
# Label each line and add a legend.
#
# Bonus: plot a second histogram for expenditure_per_student side by side.

if plt:
    # YOUR CODE HERE
    pass


In [ ]:
# 🏋️ BONUS EXERCISE
# Create a Q-Q (quantile-quantile) plot for graduation_rate.
# A Q-Q plot compares the distribution of your data to a normal distribution.
# If the points fall on the diagonal line, the data is approximately normal.
#
# Tip: use scipy.stats.probplot(data, dist='norm', plot=plt)

if plt:
    # YOUR CODE HERE
    pass


## 11. Hypothesis testing: set up H₀ and H₁

**Question:**

> Do private and non-private universities have different average graduation rates?

**Hypotheses:**

$$ \text{Denote } \mu_{private} \text{ as the mean graduation rate of private universities} $$

$$ \text{Denote } \mu_{not\ private} \text{ as the mean graduation rate of non-private universities} $$

Mathematically, we can write our hypothesis as

$$H_0: \mu_{private} - \mu_{not\ private} = 0$$

$$H_1: \mu_{private} - \mu_{not\ private} \neq 0$$





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Step 1: Create two groups by filtering
private_grad     = df[df["private"] == 1]["graduation_rate"].dropna()
non_private_grad = df[df["private"] == 0]["graduation_rate"].dropna()

# Step 2: Calculate summary statistics for each group
print(f"Private universities     : n={len(private_grad)}, mean={private_grad.mean():.4f}")
print(f"Non-private universities : n={len(non_private_grad)}, mean={non_private_grad.mean():.4f}")
print(f"Observed difference      : {private_grad.mean() - non_private_grad.mean():.4f}")
```

**Tips:**
- Always look at the descriptive statistics first — before running any formal test
- A large observed difference doesn't automatically mean statistical significance (sample size matters!)
- `.dropna()` removes any missing values before computing statistics

</details>

In [ ]:
# 🏋️ YOUR TURN
# Create two groups from df:
#   - private_grad     : graduation_rate for private universities (private == 1)
#   - non_private_grad : graduation_rate for non-private universities (private == 0)
#
# Print the mean graduation rate and sample size (n) for each group.
# Also print the observed difference in means.



In [ ]:
# 🏋️ YOUR TURN
# Present the average graduation rate for private and non-private universities
# as a horizontal bar chart for easy comparison.

if plt:
    # YOUR CODE HERE
    pass


#### 💬 Discussion

* Do we have enough evidence to suggest the mean graduation rate is different? 
* Where does it land in the context of building our machine learning model?


---


[Your comments here]



---

## 12. Welch's t-test in Python

Welch's t-test is a standard statistical test for comparing two group means.

Use it when you want to test whether two groups have different average values, **without assuming equal variance**.

| Component | Meaning |
|-----------|--------|
| t-statistic | How many standard errors the means are apart |
| p-value | Probability of seeing this difference (or larger) if H₀ were true |
| α = 0.05 | Common significance threshold |





<details>
<summary>💡 Hint — click to expand</summary>

```python
from scipy import stats

# Welch's t-test — equal_var=False means we don't assume equal variance
t_stat, p_value = stats.ttest_ind(
    private_grad,
    non_private_grad,
    equal_var=False   # <-- this is what makes it Welch's (not Student's) t-test
)

alpha = 0.05
print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_value:.4f}")

if p_value < alpha:
    print("Result: Reject H₀ — the difference in means is statistically significant.")
else:
    print("Result: Fail to reject H₀ — no statistically significant difference found.")
```

**Tips:**
- `equal_var=False` uses Welch's correction — safer when group sizes or variances differ
- The p-value is NOT the probability that H₀ is true — it is the probability of the data *given* H₀
- A small p-value means the data is unlikely under H₀, so we reject H₀
- Statistical significance ≠ practical significance — always check effect size too

</details>

In [ ]:
# 🏋️ YOUR TURN
# Run Welch's t-test to compare graduation rates between private and non-private universities.
#
# Use: stats.ttest_ind(group1, group2, equal_var=False)
# It returns: (t_statistic, p_value)
#
# Then print:
#   - the t-statistic
#   - the p-value
#   - a plain-English interpretation (reject / fail to reject H₀)

alpha = 0.05



#### 💬 How do we interpret the test result?

Discuss and mark down your comments

---


[Your comments here]



---

## 13. Chi-square test in Python

A chi-square test checks whether two **categorical** variables are associated.

Here we test:

* `private`
* `strong_outcome`

Null hypothesis:

$$H_0: \text{private status and strong outcome are independent}$$

In plain English, we want to know: does being a private university make a difference in delivering a strong outcome (i.e. higher graduation rate)?


<details>
<summary>💡 Hint — Step 1: Create the contingency table</summary>

```python
# pd.crosstab creates a contingency table — counts of combinations
contingency = pd.crosstab(
    df["private"],        # rows
    df["strong_outcome"], # columns
    rownames=["Private"],
    colnames=["Strong Outcome"]
)

print(contingency)
```

**Tips:**
- A contingency table shows counts for every combination of two categorical variables
- Look at the table first — do the proportions look different between private and non-private?
- `margins=True` in `pd.crosstab()` adds row/column totals

</details>

In [ ]:
# 🏋️ YOUR TURN — Step 1
# Create a contingency table (cross-tabulation) of private vs strong_outcome.
# Use pd.crosstab()
# Store it as 'contingency'



<details>
<summary>💡 Hint — Step 2: Run the chi-square test</summary>

```python
from scipy.stats import chi2_contingency

# chi2_contingency returns: chi2 stat, p-value, degrees of freedom, expected frequencies
chi2_stat, p_value, dof, expected = chi2_contingency(contingency)

alpha = 0.05
print(f"Chi-square statistic : {chi2_stat:.4f}")
print(f"Degrees of freedom   : {dof}")
print(f"p-value              : {p_value:.4f}")

if p_value < alpha:
    print("Result: Reject H₀ — private status and strong outcome are NOT independent.")
else:
    print("Result: Fail to reject H₀ — no significant association found.")
```

**Tips:**
- `chi2_contingency` assumes expected cell counts ≥ 5 — print `expected` to verify
- Degrees of freedom = (rows - 1) × (columns - 1)
- For a 2×2 table, `correction=True` (Yates' correction) is applied by default for small samples

</details>

In [ ]:
# 🏋️ YOUR TURN — Step 2
# Run the chi-square test of independence on the contingency table.
# Use: chi2_contingency(contingency)
# It returns: (chi2_stat, p_value, degrees_of_freedom, expected_frequencies)
#
# Print all four returned values and write a plain-English interpretation.



### 💬 Discussion

* What is your interpretation of the test result?
* How do we associate the result to building our machine learning model?

---

[Your comments here]

---

## 14. Interpret hypothesis tests

When you explain a test, include:

1. the question
2. the null hypothesis
3. the p-value
4. the plain-English meaning
5. the caution: statistical evidence is not the same as causal proof





<details>
<summary>💡 Hint — Writing a structured interpretation</summary>

```python
alpha = 0.05

# Retrieve the p-value from your t-test (cell 12) and chi-square test (cell 13)
# t_pvalue   = p-value from stats.ttest_ind()
# chi_pvalue = p-value from chi2_contingency()

def interpret_test(test_name, h0, p_value, alpha=0.05):
    decision = "REJECT H₀" if p_value < alpha else "FAIL TO REJECT H₀"
    print(f"Test          : {test_name}")
    print(f"H₀            : {h0}")
    print(f"p-value       : {p_value:.4f}")
    print(f"Decision      : {decision} (α = {alpha})")
    print(f"Caution       : Statistical significance does not imply causation.")
    print()

interpret_test(
    "Welch's t-test",
    h0="Mean graduation rate is equal for private and non-private universities",
    p_value=t_pvalue
)
```

</details>

In [ ]:
# 🏋️ YOUR TURN
# Write a structured interpretation function for hypothesis tests.
# It should print: test name, H₀, p-value, decision, and a caution note.
#
# Call it for both the t-test and the chi-square test.

alpha = 0.05

# YOUR CODE HERE


## 15. Correlation

Correlation measures how two numeric variables move together.

- **+1**: perfect positive correlation
- **0**: no linear relationship
- **-1**: perfect negative correlation

> **💬 Demo prompt:** Which feature has the strongest relationship with graduation rate?





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Select the numeric columns you care about
num_cols = ["graduation_rate", "expenditure_per_student", "acceptance_rate", "enrolment_rate"]

# Pearson correlation matrix
corr_matrix = df[num_cols].corr()

# Show correlations with graduation_rate only (sorted by absolute value)
corr_with_target = (
    corr_matrix["graduation_rate"]
    .drop("graduation_rate")    # remove self-correlation
    .sort_values(key=abs, ascending=False)
)

print(corr_with_target)

# Optionally plot a heatmap
if plt:
    import seaborn as sns  # seaborn makes heatmaps easy
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Correlation Matrix")
    plt.show()
```

**Tips:**
- `.corr()` computes Pearson correlation by default — use `method='spearman'` for non-normal data
- Sort by absolute value to find the strongest relationships regardless of direction
- High correlation ≠ causation (see Section 16!)

</details>

In [ ]:
# 🏋️ YOUR TURN
# Calculate the correlation between graduation_rate and all other numeric columns.
# Sort by absolute correlation value so the strongest relationships appear first.
#
# Bonus: plot a heatmap of the full correlation matrix.

num_cols = ["graduation_rate"]  # add more columns here



## 16. Correlation is not causation

Spend per student is correlated with graduation rate.

That does **not** prove that increasing spend *causes* higher graduation rates. Other factors may explain both.

> **💬 Discussion prompt:** What extra data would help us make a stronger causal claim?





<details>
<summary>💡 Hint — click to expand</summary>

```python
if plt:
    fig, ax = plt.subplots(figsize=(8, 5))

    # Scatter plot: x = expenditure per student, y = graduation rate
    ax.scatter(
        df["expenditure_per_student"],
        df["graduation_rate"],
        alpha=0.4,        # transparency so overlapping points are visible
        color="steelblue"
    )

    # Add a trend line using numpy polyfit
    x = df["expenditure_per_student"].dropna()
    y = df["graduation_rate"].dropna()
    # Align lengths if needed
    m, b = np.polyfit(x, y, 1)  # degree=1 = linear
    ax.plot(x, m * x + b, color="red", linewidth=2, label=f"Trend (slope={m:.4f})")

    ax.set_xlabel("Expenditure per Student")
    ax.set_ylabel("Graduation Rate")
    ax.set_title("Spend vs Graduation Rate — correlation, not causation")
    ax.legend()
    plt.tight_layout()
    plt.show()
```

**Tips:**
- `alpha=0.4` in scatter makes overlapping points visible (lower = more transparent)
- A trend line helps visually judge the direction and strength of the relationship
- Add annotations or colour by `private` to see if the relationship differs by group

</details>

In [ ]:
# 🏋️ YOUR TURN
# Create a scatter plot of expenditure_per_student (x) vs graduation_rate (y).
# Add a linear trend line.
# Add axis labels and a descriptive title.
#
# Bonus: colour the points by the 'private' column to see if patterns differ.

if plt:
    # YOUR CODE HERE
    pass


## 17. Simpson's paradox check

Simpson's paradox happens when an **overall pattern reverses after grouping**.

This dataset does not show a perfect reversal in this comparison, but it does show why subgroup checks matter.

> **💬 Think about it:** Can you think of a real-world example where the overall average is misleading?





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Group by region and compute summary stats
regional = (
    df.groupby("region")
    .agg(
        n_universities=("graduation_rate", "count"),
        avg_spend=("expenditure_per_student", "mean"),
        avg_graduation=("graduation_rate", "mean")
    )
    .round(4)
    .sort_values("avg_graduation", ascending=False)
)

print(regional)
```

**Tips:**
- `groupby().agg()` with a dictionary lets you apply different aggregations to different columns
- Named aggregations `col=(source_col, func)` make the output readable
- Compare the overall average with the group-level averages — do they tell the same story?

</details>

In [ ]:
# 🏋️ YOUR TURN
# Use groupby to calculate for each region:
#   - number of universities
#   - average spend per student
#   - average graduation rate
#
# Sort by average graduation rate (descending).
# Compare the top and bottom regions — does spending pattern match graduation rank?



## 18. Grouped comparison

Now split the acceptance-rate comparison by region.

> **💬 Interpretation prompt:** Does the subgroup view tell the same story as the aggregate view?





<details>
<summary>💡 Hint — click to expand</summary>

```python
# Group by region AND high_acceptance to compare acceptance groups within each region
grouped = (
    df.groupby(["region", "high_acceptance"])["graduation_rate"]
    .mean()
    .unstack("high_acceptance")  # pivot high_acceptance into columns
    .rename(columns={0: "low_acceptance", 1: "high_acceptance"})
    .round(4)
)

print(grouped)

# Visualise
if plt:
    grouped.plot(kind="bar", figsize=(10, 5))
    plt.title("Graduation Rate by Region and Acceptance Group")
    plt.ylabel("Mean Graduation Rate")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
```

**Tips:**
- `unstack()` pivots one level of a MultiIndex into columns — useful for side-by-side comparison
- If the high_acceptance group has a *lower* graduation rate within some regions than the aggregate, that's a hint of Simpson's paradox
- Always check sample sizes in each cell — small groups make comparisons unreliable

</details>

In [ ]:
# 🏋️ YOUR TURN
# Compare mean graduation_rate split by region AND high_acceptance group.
#
# Use groupby(["region", "high_acceptance"]) and .mean()
# Then unstack to show a side-by-side table.
#
# Optional: plot as a grouped bar chart.



## 19. Final activity

Write a short recommendation for a university support director.

Include:

1. one statistical measure
2. one probability or conditional probability
3. one hypothesis test result
4. one correlation-versus-causation warning
5. one subgroup or Simpson's paradox warning





<details>
<summary>💡 Hint — Structuring your recommendation</summary>

You can write your recommendation in a markdown cell. Here is a template:

---
**Recommendation for University Support Director**

**1. Statistical measure:** The median graduation rate across all universities is _X_%.

**2. Probability:** The probability of a strong outcome is _P(A)_ overall. For private universities this rises to _P(A|private)_.

**3. Hypothesis test:** A Welch's t-test (p = _value_) suggests that [private / non-private] universities have [significantly / not significantly] different graduation rates.

**4. Correlation caution:** While spend per student correlates with graduation rate (r = _value_), we cannot conclude that increasing spend *causes* higher graduation rates without controlling for confounders such as student demographics.

**5. Subgroup warning:** The aggregate comparison masks regional variation. In [region X], high-acceptance universities actually perform *below* low-acceptance universities — a warning sign of Simpson's paradox.

---

</details>

In [ ]:
# 🏋️ YOUR TURN
# Use this cell to pull together the key numbers for your recommendation.
# For example:
#   - Overall median graduation rate
#   - P(strong outcome) overall and P(strong outcome | private)
#   - t-test p-value
#   - Pearson correlation: spend vs graduation rate



### 📝 My Recommendation

_Write your recommendation here in plain English — imagine explaining to a non-technical stakeholder._

---

**1. Statistical measure:**

**2. Probability:**

**3. Hypothesis test result:**

**4. Correlation-versus-causation warning:**

**5. Subgroup / Simpson's paradox warning:**

---

## 20. Save our dataset for our next section

* Save the processed dataset so we can pick up where we left off in the next module.


<details>
<summary>💡 Hint — click to expand</summary>

```python
# Save the processed dataframe to a CSV file
output_path = "university_processed.csv"

df.to_csv(output_path, index=False)  # index=False avoids writing the row numbers

print(f"Saved {df.shape[0]} rows × {df.shape[1]} columns to '{output_path}'")
```

**Tips:**
- `index=False` prevents pandas from writing the integer row index as an extra column
- Consider saving to a subfolder (e.g., `data/processed/`) to keep your workspace organised
- You can also save to `.parquet` for faster reading in future: `df.to_parquet('file.parquet')`

</details>

In [ ]:
# 🏋️ YOUR TURN
# Save the final df to a CSV file called 'university_processed.csv'
# Use index=False
# Print a confirmation message with the number of rows and columns saved



## Key takeaways

* Mathematical notation is a compact way to describe statistical ideas.
* Probability can be estimated from data by counting rows.
* Statistical measures describe centre and spread.
* Hypothesis tests compare the observed result with what might happen under a null assumption.
* Welch's t-test and the chi-square test are common Python tools for hypothesis testing.
* Correlation is not causation.
* Simpson's paradox is a reason to check subgroups before trusting aggregate results.




---

### Final caveat

#### 1. **Averages Hide Complexity**
The mean alone is insufficient. You must understand the higher moments (mean, variance, skewness, kurtosis) to get a complete picture of your data's shape and behaviour.

**Real-world example:** A policy with mean outcome = 10 minutes is NOT the same whether all observations are between 8-12 minutes or some are 2 minutes and others are 95 minutes.

#### 2. **Visual Inspection is Your First Defence**
Before running any complex models:
- Plot a histogram + overlay the normal curve
- Create a Q-Q plot
- Look at the data with fresh eyes

Visual methods often reveal problems that formal tests miss and are interpretable to non-technical stakeholders.

#### 3. **Normality Testing Guides Model Selection**
- **If data is normal:** Parametric methods (linear regression, t-tests) are appropriate and efficient
- **If data is non-normal:** Consider transformations, non-parametric alternatives, or robust methods
- **Large samples:** Central Limit Theorem provides some protection against non-normality

#### 4. **Non-Normality ≠ Model Failure**
Non-normal data doesn't mean your model will fail, especially with:
- Large sample sizes (n > 500)
- Robust algorithms (tree-based models)
- Proper diagnostics and residual checking

However, it DOES mean you must be more careful and validate assumptions.

#### 5. **Context Matters More Than Rules**
The 'best' approach depends on:
- Your sample size
- Your model type
- The stakes of your decision (Is this resource allocation? Clinical diagnosis? Budget forecasting?)
- Your audience's technical understanding

---

### Practical Workflow for any data science project

```
1. Load your data
   ↓
2. Compute the four moments (mean, var, skewness, kurtosis)
   ↓
3. Create a histogram + overlay normal curve → Do they match?
   ↓
4. Create a Q-Q plot → Do points fall on the diagonal?
   ↓
5. Decision point:
   ├─ If YES to both → Data is likely normal, proceed with parametric methods
   └─ If NO to either →
       ├─ Consider transformation (log, Box-Cox)
       ├─ Or use non-parametric methods
       └─ Or document the violation and proceed with caution
   ↓
6. Check your model's residuals using the same visual methods
```

---

### Decision Tree: What Should You Do?

```
Is your data normally distributed? (Visual check)
│
├─ YES → Use parametric methods (regression, t-test, ANOVA)
│         BUT: Still check residuals after modelling!
│
└─ NO → Three options:
         1. Transform the data (log, Box-Cox)
            Then test normality again
         
         2. Use non-parametric methods
            (Mann-Whitney U, Spearman correlation, tree models)
         
         3. Proceed with parametric methods
            BUT document the assumption violation
            AND use robust standard errors
```

---

### Common Mistakes to Avoid

❌ **Mistake 1:** Using only the mean to describe your data
✓ **Better:** Always report mean ± standard deviation AND median ± IQR

❌ **Mistake 2:** Assuming normality without checking
✓ **Better:** Always visualise first (histogram, Q-Q plot)

❌ **Mistake 3:** Automatically applying transformations
✓ **Better:** Check if transformation actually helps your downstream model

❌ **Mistake 4:** Reporting transformed model coefficients without backtransforming
✓ **Better:** If you log-transformed for modelling, exponentiate predictions for interpretation

❌ **Mistake 5:** Ignoring outliers because "the test says they're expected"
✓ **Better:** Investigate outliers (data quality issues? real phenomenon?)

---